# Functions (SPINRec-style shared utilities)

> Shared components for data parsing, models (MLP/VAE/NCF), training, and DVF/SIF valuation.

## 基础定义

> 本节只包含类型与模型结构定义，不执行训练或数据处理。

In [2]:
from dataclasses import dataclass
from typing import Dict, List, Tuple
import csv
import hashlib
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DEFAULT_SPARSE = [
    "user_id", "item_id", "shop_id", "brand_id", "category_1_id",
    "city_id", "district_id", "visit_city", "time_type", "weekdays",
]
DEFAULT_DENSE = [
    "avg_price", "ctr_30", "ord_30", "total_amt_30",
    "rank_7", "rank_30", "rank_90", "hours",
]
DEFAULT_SEQ_NUM = ["price_list", "timediff_list"]


@dataclass
class PreparedData:
    dense: torch.Tensor
    sparse: Dict[str, torch.Tensor]
    labels: torch.Tensor
    sample_ids: torch.Tensor
    sparse_fields: List[str]
    dense_fields: List[str]


class RecDataset(Dataset):
    def __init__(self, prepared: PreparedData):
        self.prepared = prepared

    def __len__(self):
        return self.prepared.labels.shape[0]

    def __getitem__(self, idx: int):
        return {
            "dense": self.prepared.dense[idx],
            "labels": self.prepared.labels[idx],
            "sample_id": self.prepared.sample_ids[idx],
            "sparse": {k: v[idx] for k, v in self.prepared.sparse.items()},
        }


class SparseEncoder(nn.Module):
    def __init__(self, sparse_fields: List[str], num_buckets: int, emb_dim: int):
        super().__init__()
        self.sparse_fields = sparse_fields
        self.emb = nn.ModuleDict({k: nn.Embedding(num_buckets, emb_dim) for k in sparse_fields})

    def forward(self, sparse: Dict[str, torch.Tensor]):
        return torch.cat([self.emb[k](sparse[k]) for k in self.sparse_fields], dim=1)


class MLPRec(nn.Module):
    def __init__(self, sparse_fields, dense_dim, num_buckets, emb_dim=16):
        super().__init__()
        self.enc = SparseEncoder(sparse_fields, num_buckets, emb_dim)
        in_dim = dense_dim + emb_dim * len(sparse_fields)
        self.net = nn.Sequential(
            nn.Linear(in_dim, 128), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 1),
        )

    def forward(self, dense, sparse):
        x = torch.cat([dense, self.enc(sparse)], dim=1)
        return self.net(x).squeeze(1)


class NCFRec(nn.Module):
    def __init__(self, num_buckets, emb_dim=32):
        super().__init__()
        self.ug = nn.Embedding(num_buckets, emb_dim)
        self.ig = nn.Embedding(num_buckets, emb_dim)
        self.um = nn.Embedding(num_buckets, emb_dim)
        self.im = nn.Embedding(num_buckets, emb_dim)
        self.mlp = nn.Sequential(
            nn.Linear(emb_dim * 2, 128), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(128, 64), nn.ReLU(),
        )
        self.out = nn.Linear(emb_dim + 64, 1)

    def forward(self, dense, sparse):
        u = sparse["user_id"]
        i = sparse["item_id"]
        gmf = self.ug(u) * self.ig(i)
        m = self.mlp(torch.cat([self.um(u), self.im(i)], dim=1))
        return self.out(torch.cat([gmf, m], dim=1)).squeeze(1)


class VAERec(nn.Module):
    def __init__(self, sparse_fields, dense_dim, num_buckets, emb_dim=16, latent_dim=32):
        super().__init__()
        self.enc_sparse = SparseEncoder(sparse_fields, num_buckets, emb_dim)
        in_dim = dense_dim + emb_dim * len(sparse_fields)
        self.encoder = nn.Sequential(nn.Linear(in_dim, 128), nn.ReLU(), nn.Linear(128, 64), nn.ReLU())
        self.mu = nn.Linear(64, latent_dim)
        self.logvar = nn.Linear(64, latent_dim)
        self.decoder = nn.Sequential(nn.Linear(latent_dim, 64), nn.ReLU(), nn.Linear(64, 1))

    def forward(self, dense, sparse):
        x = torch.cat([dense, self.enc_sparse(sparse)], dim=1)
        h = self.encoder(x)
        mu = self.mu(h)
        logvar = self.logvar(h)
        z = mu + torch.randn_like(mu) * torch.exp(0.5 * logvar)
        logit = self.decoder(z).squeeze(1)
        return logit, mu, logvar

### seed_everything

> 作用：固定随机种子，保证实验可复现。

In [3]:
def seed_everything(seed: int = 42):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

### load_fields

> 作用：读取字段定义文件并建立列名到索引的映射。

In [4]:
def load_fields(fields_csv: str) -> Tuple[List[str], Dict[str, int]]:
    names = []
    with open(fields_csv, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        for i, row in enumerate(reader):
            if i == 0 or not row:
                continue
            names.append(row[0].strip())
    return names, {n: i for i, n in enumerate(names)}

### _sf

> 作用：安全地将字符串转为浮点数（空值和异常统一返回0）。

In [5]:
def _sf(x: str) -> float:
    t = (x or "").strip()
    if t in ("", "-1"):
        return 0.0
    try:
        return float(t)
    except Exception:
        return 0.0

### _hash_bucket

> 作用：将离散值稳定哈希到固定桶，控制词表大小。

In [6]:
def _hash_bucket(x: str, mod: int) -> int:
    t = (x or "-1").strip() or "-1"
    h = hashlib.md5(t.encode("utf-8")).hexdigest()[:8]
    return int(h, 16) % mod

### _seq_mean

> 作用：将分号分隔序列聚合为均值特征。

In [7]:
def _seq_mean(x: str) -> float:
    t = (x or "").strip()
    if not t:
        return 0.0
    parts = [p for p in t.split(";") if p and p != "-1"]
    if not parts:
        return 0.0
    vals = [_sf(p) for p in parts]
    return sum(vals) / max(len(vals), 1)

### parse_ele_data

> 作用：将 Eleme 原始样本解析为 `PreparedData`，用于训练与评估。

In [8]:
def parse_ele_data(
    data_file: str,
    fields_csv: str,
    max_rows: int = 20000,
    sparse_fields: List[str] = None,
    dense_fields: List[str] = None,
    seq_num_fields: List[str] = None,
    num_buckets: int = 50000,
) -> PreparedData:
    names, index = load_fields(fields_csv)
    sparse_fields = [f for f in (sparse_fields or DEFAULT_SPARSE) if f in index]
    dense_base = [f for f in (dense_fields or DEFAULT_DENSE) if f in index]
    seq_num_fields = [f for f in (seq_num_fields or DEFAULT_SEQ_NUM) if f in index]
    dense_all = dense_base + [f"{f}_mean" for f in seq_num_fields]

    dense_rows = []
    sparse_rows = {k: [] for k in sparse_fields}
    labels = []
    sids = []

    with open(data_file, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        for row_id, row in enumerate(reader):
            if not row:
                continue
            if max_rows > 0 and len(labels) >= max_rows:
                break

            y = 1.0 if _sf(row[index.get("label", 0)]) > 0 else 0.0
            labels.append(y)
            sids.append(row_id)

            d = [_sf(row[index[k]]) for k in dense_base]
            d += [_seq_mean(row[index[k]]) for k in seq_num_fields]
            dense_rows.append(d)

            for k in sparse_fields:
                v = row[index[k]] if index[k] < len(row) else "-1"
                sparse_rows[k].append(_hash_bucket(v, num_buckets))

    dense = torch.tensor(dense_rows, dtype=torch.float32)
    if dense.numel() > 0:
        mean = dense.mean(0, keepdim=True)
        std = dense.std(0, keepdim=True).clamp(min=1e-6)
        dense = (dense - mean) / std

    sparse = {k: torch.tensor(v, dtype=torch.long) for k, v in sparse_rows.items()}
    labels = torch.tensor(labels, dtype=torch.float32)
    sids = torch.tensor(sids, dtype=torch.long)

    return PreparedData(
        dense=dense, sparse=sparse, labels=labels, sample_ids=sids,
        sparse_fields=sparse_fields, dense_fields=dense_all,
    )

### split_prepared

> 作用：按比例划分训练/验证集。

In [9]:
def split_prepared(prepared: PreparedData, val_ratio: float = 0.2, seed: int = 42):
    g = torch.Generator().manual_seed(seed)
    n = prepared.labels.shape[0]
    p = torch.randperm(n, generator=g)
    n_val = max(1, int(n * val_ratio))
    val_i = p[:n_val]
    tr_i = p[n_val:]

    def _sub(i: torch.Tensor) -> PreparedData:
        return PreparedData(
            dense=prepared.dense[i],
            sparse={k: v[i] for k, v in prepared.sparse.items()},
            labels=prepared.labels[i],
            sample_ids=prepared.sample_ids[i],
            sparse_fields=prepared.sparse_fields,
            dense_fields=prepared.dense_fields,
        )

    return _sub(tr_i), _sub(val_i)

### build_model

> 作用：按模型名构建 MLP/NCF/VAE 推荐器。

In [10]:
def build_model(name: str, sparse_fields: List[str], dense_dim: int, num_buckets: int):
    n = name.upper()
    if n == "MLP":
        return MLPRec(sparse_fields, dense_dim, num_buckets)
    if n == "NCF":
        if "user_id" not in sparse_fields or "item_id" not in sparse_fields:
            raise ValueError("NCF requires user_id and item_id")
        return NCFRec(num_buckets)
    if n == "VAE":
        return VAERec(sparse_fields, dense_dim, num_buckets)
    raise ValueError(f"Unknown model: {name}")

### _loss

> 作用：统一二分类损失函数（BCE with logits）。

In [11]:
def _extract_logits(out: torch.Tensor) -> torch.Tensor:
    if isinstance(out, (tuple, list)):
        return out[0]
    return out

def _loss(logits: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    logits = _extract_logits(logits)
    return F.binary_cross_entropy_with_logits(logits.view(-1), y.float().view(-1))

### _auc

> 作用：无外部依赖地计算 AUC。

In [12]:
def _auc(y_true, y_score) -> float:
    y_true = np.asarray(y_true).astype(np.int64)
    y_score = np.asarray(y_score).astype(np.float64)
    pos = int((y_true == 1).sum())
    neg = int((y_true == 0).sum())
    if pos == 0 or neg == 0:
        return 0.5
    order = np.argsort(y_score)
    ranks = np.empty_like(order, dtype=np.float64)
    ranks[order] = np.arange(1, len(y_score) + 1, dtype=np.float64)
    pos_rank_sum = ranks[y_true == 1].sum()
    auc = (pos_rank_sum - pos * (pos + 1) / 2.0) / (pos * neg)
    return float(auc)

### evaluate_model

> 作用：在 DataLoader 上评估平均损失与 AUC。

In [13]:
def evaluate_model(model: nn.Module, loader: DataLoader, device: str = DEVICE) -> Dict[str, float]:
    model.eval()
    ys = []
    ps = []
    losses = []
    with torch.no_grad():
        for dense, sparse, y, _sid in loader:
            dense = dense.to(device)
            sparse = {k: v.to(device) for k, v in sparse.items()}
            y = y.to(device)
            logits = model(dense, sparse)
            losses.append(float(_loss(logits, y).item()))
            ys.append(y.cpu().numpy().reshape(-1))
            ps.append(torch.sigmoid(logits).cpu().numpy().reshape(-1))
    y_all = np.concatenate(ys) if ys else np.array([0, 1])
    p_all = np.concatenate(ps) if ps else np.array([0.5, 0.5])
    return {"loss": float(np.mean(losses) if losses else 0.0), "auc": _auc(y_all, p_all)}

### _flat

> 作用：将模型参数梯度展平为单向量，便于相似度计算。

In [14]:
def _flat(grads: List[torch.Tensor]) -> torch.Tensor:
    if not grads:
        return torch.zeros(1)
    return torch.cat([g.reshape(-1) for g in grads])

### val_grad

> 作用：计算验证集平均损失对参数的梯度向量。

In [15]:
def val_grad(model: nn.Module, loader: DataLoader, device: str = DEVICE) -> torch.Tensor:
    model.eval()
    acc = None
    n = 0
    for dense, sparse, y, _sid in loader:
        dense = dense.to(device)
        sparse = {k: v.to(device) for k, v in sparse.items()}
        y = y.to(device)
        logits = _extract_logits(model(dense, sparse))
        loss = _loss(logits, y)
        grads = torch.autograd.grad(loss, [p for p in model.parameters() if p.requires_grad], retain_graph=False, allow_unused=True)
        g = _flat([x.detach().cpu() for x in grads if x is not None])
        acc = g if acc is None else (acc + g)
        n += 1
    if acc is None:
        return torch.zeros(1)
    return acc / max(n, 1)

### sample_grad

> 作用：获取单样本损失对参数的梯度向量。

In [16]:
def sample_grad(model: nn.Module, dense: torch.Tensor, sparse: Dict[str, torch.Tensor], y: torch.Tensor, device: str = DEVICE) -> torch.Tensor:
    model.eval()
    dense = dense.to(device)
    sparse = {k: v.to(device) for k, v in sparse.items()}
    y = y.to(device)
    logits = _extract_logits(model(dense, sparse))
    loss = _loss(logits, y)
    grads = torch.autograd.grad(loss, [p for p in model.parameters() if p.requires_grad], retain_graph=False, allow_unused=True)
    return _flat([x.detach().cpu() for x in grads if x is not None])

### compute_sif

> 作用：计算 SIF（样本梯度与验证梯度的内积）样本价值分数。

In [17]:

def compute_sif(
    model: nn.Module,
    val_loader: DataLoader,
    train_loader_eval: DataLoader,
    max_samples: int = 256,
    device: str = DEVICE,
):
    vg = val_grad(model, val_loader, device=device)
    scores = {}
    c = 0
    for batch in train_loader_eval:
        dense, sparse, y, sid = _unpack_batch(batch)
        bs = dense.shape[0]
        for i in range(bs):
            g = sample_grad(
                model,
                dense[i : i + 1],
                {k: v[i : i + 1] for k, v in sparse.items()},
                y[i : i + 1],
                device=device,
            )
            s = float(torch.dot(g, vg).item()) if g.numel() == vg.numel() else 0.0
            scores[int(sid[i].item())] = s
            c += 1
            if c >= max_samples:
                return scores
    return scores

### compute_dvf_stage

> 作用：计算 DVF 的分阶段累计价值分数。

In [18]:
def compute_dvf_stage(
    model_builder,
    ckpt_paths: List[str],
    train_loader: DataLoader,
    val_loader: DataLoader,
    max_batches: int = 50,
    device: str = DEVICE,
) -> pd.DataFrame:
    scores = {}
    for p in ckpt_paths:
        model = model_builder()
        model.load_state_dict(torch.load(p, map_location=device))
        model.to(device)
        vg = val_grad(model, val_loader, device=device)
        c = 0
        for dense, sparse, y, sid in train_loader:
            bs = dense.shape[0]
            for i in range(bs):
                g = sample_grad(
                    model,
                    dense[i : i + 1],
                    {k: v[i : i + 1] for k, v in sparse.items()},
                    y[i : i + 1],
                    device=device,
                )
                s = float(torch.dot(g, vg).item()) if g.numel() == vg.numel() else 0.0
                k = int(sid[i].item())
                scores[k] = scores.get(k, 0.0) + s
            c += 1
            if c >= max_batches:
                break
    out = [{"sample_id": k, "dvf": v} for k, v in scores.items()]
    return pd.DataFrame(out).sort_values("dvf", ascending=False).reset_index(drop=True)

### make_loaders

> 作用：将 PreparedData 包装为 DataLoader。

In [19]:
def make_loaders(train_d: PreparedData, val_d: PreparedData, batch_size: int = 1024):
    train_loader = DataLoader(RecDataset(train_d), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(RecDataset(val_d), batch_size=batch_size, shuffle=False)
    return train_loader, val_loader

### save_table

> 作用：保存表格结果到 CSV，并确保目录存在。

In [20]:
def save_table(df: pd.DataFrame, path: str):
    p = Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(p, index=False)
    print(f"saved: {p}")

### _recursive_filter_binary_interactions

> 作用：递归过滤低频用户与低频物品，得到更稳定的隐式反馈子集。

In [21]:
def _recursive_filter_binary_interactions(
    df: pd.DataFrame,
    user_col: str,
    item_col: str,
    min_user_count: int = 5,
    min_item_count: int = 5,
) -> pd.DataFrame:
    cur = df.copy()
    while True:
        nu0 = cur[user_col].nunique()
        ni0 = cur[item_col].nunique()
        uc = cur.groupby(user_col)[item_col].count()
        keep_u = set(uc[uc >= min_user_count].index.tolist())
        cur = cur[cur[user_col].isin(keep_u)]
        ic = cur.groupby(item_col)[user_col].count()
        keep_i = set(ic[ic >= min_item_count].index.tolist())
        cur = cur[cur[item_col].isin(keep_i)]
        nu1 = cur[user_col].nunique()
        ni1 = cur[item_col].nunique()
        if nu1 == nu0 and ni1 == ni0:
            break
    return cur.reset_index(drop=True)

### _encode_ids

> 作用：将用户与物品 ID 映射到连续整数索引。

In [22]:
def _encode_ids(df: pd.DataFrame, user_col: str, item_col: str):
    u_vals = sorted(df[user_col].unique().tolist())
    i_vals = sorted(df[item_col].unique().tolist())
    u_map = {u: idx for idx, u in enumerate(u_vals)}
    i_map = {i: idx for idx, i in enumerate(i_vals)}
    out = df.copy()
    out["uid"] = out[user_col].map(u_map).astype(np.int64)
    out["iid"] = out[item_col].map(i_map).astype(np.int64)
    return out, u_map, i_map

### _build_user_item_matrix

> 作用：构建用户-物品二值交互矩阵。

In [23]:
def _build_user_item_matrix(df_uid_iid: pd.DataFrame, num_users: int, num_items: int):
    mat = np.zeros((num_users, num_items), dtype=np.int8)
    mat[df_uid_iid["uid"].values, df_uid_iid["iid"].values] = 1
    return mat

### _user_level_split

> 作用：按用户维度划分训练用户与测试用户。

In [24]:
def _user_level_split(users, test_ratio: float = 0.2, seed: int = 42):
    rng = np.random.RandomState(seed)
    users = np.array(users)
    rng.shuffle(users)
    n_test = max(1, int(len(users) * test_ratio))
    test_u = set(users[:n_test].tolist())
    train_u = set(users[n_test:].tolist())
    return train_u, test_u

### _static_test_pos_neg

> 作用：为测试用户采样静态正负样本对。

In [25]:
def _static_test_pos_neg(
    test_df: pd.DataFrame,
    full_matrix,
    num_items: int,
    neg_per_pos: int = 5,
    seed: int = 42,
):
    rng = np.random.RandomState(seed)
    user_pos = test_df.groupby("uid")["iid"].apply(list).to_dict()
    pos_pairs = []
    neg_pairs = []
    for u, pos_items in user_pos.items():
        pos_items = list(set(pos_items))
        for i in pos_items:
            pos_pairs.append((u, i))
            neg_need = neg_per_pos
            while neg_need > 0:
                j = int(rng.randint(0, num_items))
                if full_matrix[u, j] == 0:
                    neg_pairs.append((u, j))
                    neg_need -= 1
    return np.array(pos_pairs, dtype=np.int64), np.array(neg_pairs, dtype=np.int64)

### _item_popularity

> 作用：统计物品流行度，用于后续负采样策略。

In [26]:
def _item_popularity(train_df: pd.DataFrame, num_items: int):
    pop = np.zeros(num_items, dtype=np.float32)
    vc = train_df["iid"].value_counts()
    for iid, c in vc.items():
        pop[int(iid)] = float(c)
    s = float(pop.sum())
    if s > 0:
        pop = pop / s
    else:
        pop[:] = 1.0 / max(num_items, 1)
    return pop

### _build_prepared_from_uid_iid

> 作用：由 uid/iid 构造 MLP/NCF/VAE 可训练的 PreparedData。

In [27]:
def _build_prepared_from_uid_iid(
    train_df: pd.DataFrame,
    num_items: int,
    neg_ratio: int = 4,
    seed: int = 42,
) -> PreparedData:
    rng = np.random.RandomState(seed)
    rows = []
    sid = 0
    user_pos = train_df.groupby("uid")["iid"].apply(set).to_dict()
    for u, pos_set in user_pos.items():
        for i in pos_set:
            rows.append((u, i, 1.0, sid))
            sid += 1
            for _ in range(neg_ratio):
                j = int(rng.randint(0, num_items))
                while j in pos_set:
                    j = int(rng.randint(0, num_items))
                rows.append((u, j, 0.0, sid))
                sid += 1
    arr = np.array(rows, dtype=np.float32)
    u = arr[:, 0].astype(np.int64)
    i = arr[:, 1].astype(np.int64)
    y = arr[:, 2].astype(np.float32)
    s = arr[:, 3].astype(np.int64)
    dense = torch.zeros((len(rows), 1), dtype=torch.float32)
    sparse = {
        "user_id": torch.tensor(u, dtype=torch.long),
        "item_id": torch.tensor(i, dtype=torch.long),
    }
    return PreparedData(
        dense=dense,
        sparse=sparse,
        labels=torch.tensor(y, dtype=torch.float32),
        sample_ids=torch.tensor(s, dtype=torch.long),
        sparse_fields=["user_id", "item_id"],
        dense_fields=["dummy_dense"],
    )

### parse_quick_dataset

> 作用：读取 quick_run_data 下多种公开数据格式并统一为 user/item/label。

In [28]:
def parse_quick_dataset(path: str, dataset: str = "Pinterest") -> pd.DataFrame:
    ds = dataset.strip().lower()
    p = Path(path)
    if ds == "pinterest":
        df = pd.read_csv(p)
        cols = [c.lower() for c in df.columns]
        if "user" in cols and "item" in cols:
            user_c = df.columns[cols.index("user")]
            item_c = df.columns[cols.index("item")]
        else:
            user_c, item_c = df.columns[:2]
        out = pd.DataFrame({"user": df[user_c], "item": df[item_c], "label": 1.0})
        return out
    if ds == "yahoo":
        df = pd.read_csv(p)
        cols = {c.lower(): c for c in df.columns}
        u = cols.get("user") or list(df.columns)[0]
        i = cols.get("item") or list(df.columns)[1]
        r = cols.get("rating") or list(df.columns)[2]
        out = pd.DataFrame({"user": df[u], "item": df[i], "label": (df[r] > 0).astype(float)})
        return out
    if ds in ["ml1m", "movielens", "movie_lens"]:
        rows = []
        with open(p, "r", encoding="utf-8") as f:
            for line in f:
                s = line.strip()
                if not s:
                    continue
                sp = s.split("::")
                if len(sp) >= 3:
                    rows.append((sp[0], sp[1], float(sp[2])))
        df = pd.DataFrame(rows, columns=["user", "item", "rating"])
        df["label"] = (df["rating"] >= 4.0).astype(float)
        return df[["user", "item", "label"]]
    raise ValueError(f"Unsupported quick dataset: {dataset}")

### prepare_quick_run_data_spinrec_style

> 作用：按 SPINRec 风格处理 quick_run 数据，并产出训练与评估所需完整结构。

In [29]:
def prepare_quick_run_data_spinrec_style(
    quick_data_path: str,
    dataset_name: str = "Pinterest",
    min_user_count: int = 5,
    min_item_count: int = 5,
    user_test_ratio: float = 0.2,
    neg_per_pos: int = 5,
    train_neg_ratio: int = 4,
    seed: int = 42,
):
    raw = parse_quick_dataset(quick_data_path, dataset=dataset_name)
    raw = raw[raw["label"] > 0].copy()
    raw = _recursive_filter_binary_interactions(
        raw.rename(columns={"user": "u", "item": "i"}),
        user_col="u",
        item_col="i",
        min_user_count=min_user_count,
        min_item_count=min_item_count,
    ).rename(columns={"u": "user", "i": "item"})
    enc, u_map, i_map = _encode_ids(raw, "user", "item")
    num_users = len(u_map)
    num_items = len(i_map)
    full_mat = _build_user_item_matrix(enc, num_users, num_items)
    train_u, test_u = _user_level_split(np.array(sorted(enc["uid"].unique())), test_ratio=user_test_ratio, seed=seed)
    train_df = enc[enc["uid"].isin(train_u)].copy()
    test_df = enc[enc["uid"].isin(test_u)].copy()
    test_pos, test_neg = _static_test_pos_neg(
        test_df,
        full_mat,
        num_items,
        neg_per_pos=neg_per_pos,
        seed=seed,
    )
    pop = _item_popularity(train_df, num_items)
    prepared = _build_prepared_from_uid_iid(train_df, num_items=num_items, neg_ratio=train_neg_ratio, seed=seed)
    stats = {
        "num_users": num_users,
        "num_items": num_items,
        "num_interactions": int(enc.shape[0]),
        "train_users": int(len(train_u)),
        "test_users": int(len(test_u)),
    }
    artifacts = {
        "full_interaction_matrix": full_mat,
        "train_df_uid_iid": train_df[["uid", "iid"]].reset_index(drop=True),
        "test_df_uid_iid": test_df[["uid", "iid"]].reset_index(drop=True),
        "test_pos_pairs": test_pos,
        "test_neg_pairs": test_neg,
        "item_popularity": pop,
        "user_map": u_map,
        "item_map": i_map,
    }
    return prepared, artifacts, stats

### make_loaders（兼容调用签名）

> 作用：从单个 PreparedData 切分并返回 train_loader、train_eval_loader、val_loader。

In [30]:
def make_loaders(prepared: PreparedData, batch_size: int = 1024, val_ratio: float = 0.2, seed: int = 42):
    train_d, val_d = split_prepared(prepared, val_ratio=val_ratio, seed=seed)
    train_loader = DataLoader(RecDataset(train_d), batch_size=batch_size, shuffle=True)
    train_eval_loader = DataLoader(RecDataset(train_d), batch_size=batch_size, shuffle=False)
    val_loader = DataLoader(RecDataset(val_d), batch_size=batch_size, shuffle=False)
    return train_loader, train_eval_loader, val_loader

### evaluate_model（补充 acc 输出）

> 作用：返回 loss/acc/auc，兼容训练汇总逻辑。

In [31]:
def evaluate_model(model: nn.Module, loader: DataLoader, device: str = DEVICE) -> Dict[str, float]:
    model.eval()
    ys = []
    ps = []
    losses = []
    with torch.no_grad():
        for dense, sparse, y, _sid in loader:
            dense = dense.to(device)
            sparse = {k: v.to(device) for k, v in sparse.items()}
            y = y.to(device)
            logits = model(dense, sparse)
            losses.append(float(_loss(logits, y).item()))
            ys.append(y.cpu().numpy().reshape(-1))
            ps.append(torch.sigmoid(logits).cpu().numpy().reshape(-1))
    y_all = np.concatenate(ys) if ys else np.array([0, 1])
    p_all = np.concatenate(ps) if ps else np.array([0.5, 0.5])
    pred = (p_all >= 0.5).astype(np.int64)
    acc = float((pred == y_all.astype(np.int64)).mean())
    return {
        "loss": float(np.mean(losses) if losses else 0.0),
        "acc": acc,
        "auc": _auc(y_all, p_all),
    }

### train_model（返回 history 与 checkpoints）

> 作用：训练模型并返回历史表与分阶段检查点。

In [ ]:
def train_model(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    epochs: int = 3,
    lr: float = 1e-3,
    device: str = DEVICE,
    verbose: bool = True,
    early_stop_patience: int = 0,
    early_stop_min_delta: float = 0.0,
    save_end_only: bool = False,
    checkpoint_interval_epochs: float = 0.5,
    use_class_balance: bool = True,
    weight_decay: float = 1e-5,
    max_grad_norm: float = 5.0,
    lr_scheduler_patience: int = 2,
    lr_scheduler_factor: float = 0.5,
    min_lr: float = 1e-6,
):
    model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=float(weight_decay))
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt,
        mode="max",
        factor=float(lr_scheduler_factor),
        patience=int(lr_scheduler_patience),
        min_lr=float(min_lr),
        verbose=False,
    )

    # 可选：根据训练集正负比例做 BCE 正类加权，缓解严重不平衡下的学习偏置
    pos_weight_t = None
    if use_class_balance:
        try:
            labels_all = train_loader.dataset.prepared.labels.detach().float().cpu()
            pos = float((labels_all > 0.5).sum().item())
            neg = float((labels_all <= 0.5).sum().item())
            if pos > 0 and neg > 0:
                pw = max(1.0, min(neg / pos, 20.0))
                pos_weight_t = torch.tensor([pw], dtype=torch.float32, device=device)
        except Exception:
            pos_weight_t = None

    hist = []
    snapshots = []
    best_auc = -1e18
    best_state = None
    bad_epochs = 0

    ckpt_interval = float(checkpoint_interval_epochs)
    if ckpt_interval <= 0:
        ckpt_interval = 0.5
    next_ckpt_epoch = ckpt_interval

    for ep in range(1, epochs + 1):
        model.train()
        run_loss = 0.0
        n = 0
        for batch in train_loader:
            dense, sparse, y, _sid = _unpack_batch(batch)
            dense = dense.to(device)
            sparse = {k: v.to(device) for k, v in sparse.items()}
            y = y.to(device)
            logits = model(dense, sparse)
            if isinstance(logits, tuple):
                logits = logits[0]
            if pos_weight_t is not None:
                loss = F.binary_cross_entropy_with_logits(
                    logits.view(-1), y.float().view(-1), pos_weight=pos_weight_t
                )
            else:
                loss = _loss(logits, y)
            opt.zero_grad()
            loss.backward()
            if max_grad_norm and float(max_grad_norm) > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=float(max_grad_norm))
            opt.step()
            bs = int(y.shape[0])
            run_loss += float(loss.item()) * bs
            n += bs

        tr_loss = run_loss / max(n, 1)
        val_m = evaluate_model(model, val_loader, device=device)
        sched.step(float(val_m.get("auc", 0.0)))
        row = {
            "epoch": ep,
            "train_loss": tr_loss,
            "lr": float(opt.param_groups[0]["lr"]),
            **val_m,
        }
        hist.append(row)
        if verbose:
            print(row)

        if not save_end_only:
            if (ep + 1e-12) >= next_ckpt_epoch:
                snapshots.append({k: v.detach().cpu().clone() for k, v in model.state_dict().items()})
                while next_ckpt_epoch <= (ep + 1e-12):
                    next_ckpt_epoch += ckpt_interval

        if early_stop_patience and early_stop_patience > 0:
            cur_auc = float(val_m.get("auc", 0.0))
            if cur_auc > best_auc + float(early_stop_min_delta):
                best_auc = cur_auc
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                bad_epochs = 0
            else:
                bad_epochs += 1
                if bad_epochs >= int(early_stop_patience):
                    if verbose:
                        print({"early_stop": True, "stopped_epoch": ep, "best_auc": best_auc})
                    break

    if best_state is not None:
        model.load_state_dict(best_state)

    final_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    if save_end_only:
        snapshots = [final_state]
    else:
        snapshots.append(final_state)

    return pd.DataFrame(hist), snapshots

### compute_sif（返回 sample_id->score 字典）

> 作用：兼容评估 notebook，输出样本价值映射。

In [33]:
def compute_sif(
    model: nn.Module,
    val_loader: DataLoader,
    train_loader_eval: DataLoader,
    max_samples: int = 256,
    device: str = DEVICE,
):
    vg = val_grad(model, val_loader, device=device)
    scores = {}
    c = 0
    for dense, sparse, y, sid in train_loader_eval:
        bs = dense.shape[0]
        for i in range(bs):
            g = sample_grad(
                model,
                dense[i : i + 1],
                {k: v[i : i + 1] for k, v in sparse.items()},
                y[i : i + 1],
                device=device,
            )
            s = float(torch.dot(g, vg).item()) if g.numel() == vg.numel() else 0.0
            scores[int(sid[i].item())] = s
            c += 1
            if c >= max_samples:
                return scores
    return scores

### compute_dvf_stage（返回 total/stage1/stage2）

> 作用：兼容评估 notebook 的 DVF 分阶段输出接口。

In [34]:
def compute_dvf_stage(
    model_builder,
    checkpoints,
    train_loader_eval: DataLoader,
    val_loader: DataLoader,
    max_samples: int = 256,
    lr: float = 1e-3,
    device: str = DEVICE,
):
    stage_scores = []
    for ck in checkpoints:
        model = model_builder()
        if isinstance(ck, str):
            state = torch.load(ck, map_location="cpu")
        else:
            state = ck
        model.load_state_dict(state)
        model.to(device)
        vg = val_grad(model, val_loader, device=device)
        cur = {}
        c = 0
        for dense, sparse, y, sid in train_loader_eval:
            bs = dense.shape[0]
            for i in range(bs):
                g = sample_grad(
                    model,
                    dense[i : i + 1],
                    {k: v[i : i + 1] for k, v in sparse.items()},
                    y[i : i + 1],
                    device=device,
                )
                s = float(torch.dot(g, vg).item()) if g.numel() == vg.numel() else 0.0
                cur[int(sid[i].item())] = s
                c += 1
                if c >= max_samples:
                    break
            if c >= max_samples:
                break
        stage_scores.append(cur)
    stage1 = stage_scores[0] if len(stage_scores) > 0 else {}
    stage2 = stage_scores[1] if len(stage_scores) > 1 else {}
    ids = set(stage1.keys()) | set(stage2.keys())
    total = {k: stage1.get(k, 0.0) + stage2.get(k, 0.0) for k in ids}
    return total, stage1, stage2

### save_table（兼容 DataFrame/字典列表）

> 作用：保存 DataFrame 或记录列表到 CSV。

In [35]:
def save_table(data, path: str):
    p = Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    if isinstance(data, pd.DataFrame):
        df = data
    else:
        df = pd.DataFrame(data)
    df.to_csv(p, index=False)
    print(f"saved: {p}")

### _unpack_batch（兼容字典/元组批次）

> 作用：统一从 DataLoader 批次中提取 dense/sparse/labels/sample_id，修复训练时字符串解包问题。

In [36]:
def _unpack_batch(batch):
    if isinstance(batch, dict):
        return batch["dense"], batch["sparse"], batch["labels"], batch["sample_id"]
    if isinstance(batch, (list, tuple)) and len(batch) == 4:
        return batch[0], batch[1], batch[2], batch[3]
    raise TypeError("Unsupported batch format")

### 训练评估函数（批次兼容修复）

> 作用：覆盖 train/eval/valuation 相关函数，兼容当前 RecDataset 的字典批次格式。

In [37]:
def evaluate_model(model: nn.Module, loader: DataLoader, device: str = DEVICE) -> Dict[str, float]:
    model.eval()
    ys = []
    ps = []
    losses = []
    with torch.no_grad():
        for batch in loader:
            dense, sparse, y, _sid = _unpack_batch(batch)
            dense = dense.to(device)
            sparse = {k: v.to(device) for k, v in sparse.items()}
            y = y.to(device)
            logits = model(dense, sparse)
            if isinstance(logits, tuple):
                logits = logits[0]
            losses.append(float(_loss(logits, y).item()))
            ys.append(y.detach().cpu().numpy().reshape(-1))
            ps.append(torch.sigmoid(logits).detach().cpu().numpy().reshape(-1))
    y_all = np.concatenate(ys) if ys else np.array([0, 1])
    p_all = np.concatenate(ps) if ps else np.array([0.5, 0.5])
    pred = (p_all >= 0.5).astype(np.int64)
    acc = float((pred == y_all.astype(np.int64)).mean())
    return {
        "loss": float(np.mean(losses) if losses else 0.0),
        "acc": acc,
        "auc": _auc(y_all, p_all),
    }




def val_grad(model: nn.Module, loader: DataLoader, device: str = DEVICE) -> torch.Tensor:
    model.eval()
    acc = None
    n = 0
    for batch in loader:
        dense, sparse, y, _sid = _unpack_batch(batch)
        dense = dense.to(device)
        sparse = {k: v.to(device) for k, v in sparse.items()}
        y = y.to(device)
        logits = model(dense, sparse)
        if isinstance(logits, tuple):
            logits = logits[0]
        loss = _loss(logits, y)
        grads = torch.autograd.grad(
            loss,
            [p for p in model.parameters() if p.requires_grad],
            retain_graph=False,
            allow_unused=True,
        )
        g = _flat([x.detach().cpu() for x in grads if x is not None])
        acc = g if acc is None else (acc + g)
        n += 1
    if acc is None:
        return torch.zeros(1)
    return acc / max(n, 1)


def compute_sif(
    model: nn.Module,
    val_loader: DataLoader,
    train_loader_eval: DataLoader,
    max_samples: int = 256,
    device: str = DEVICE,
):
    vg = val_grad(model, val_loader, device=device)
    scores = {}
    c = 0
    for batch in train_loader_eval:
        dense, sparse, y, sid = _unpack_batch(batch)
        bs = dense.shape[0]
        for i in range(bs):
            g = sample_grad(
                model,
                dense[i : i + 1],
                {k: v[i : i + 1] for k, v in sparse.items()},
                y[i : i + 1],
                device=device,
            )
            s = float(torch.dot(g, vg).item()) if g.numel() == vg.numel() else 0.0
            scores[int(sid[i].item())] = s
            c += 1
            if c >= max_samples:
                return scores
    return scores


def compute_dvf_stage(
    model_builder,
    checkpoints,
    train_loader_eval: DataLoader,
    val_loader: DataLoader,
    max_samples: int = 256,
    lr: float = 1e-3,
    device: str = DEVICE,
):
    stage_scores = []
    for ck in checkpoints:
        model = model_builder()
        if isinstance(ck, str):
            state = torch.load(ck, map_location="cpu")
        else:
            state = ck
        model.load_state_dict(state)
        model.to(device)
        vg = val_grad(model, val_loader, device=device)
        cur = {}
        c = 0
        for batch in train_loader_eval:
            dense, sparse, y, sid = _unpack_batch(batch)
            bs = dense.shape[0]
            for i in range(bs):
                g = sample_grad(
                    model,
                    dense[i : i + 1],
                    {k: v[i : i + 1] for k, v in sparse.items()},
                    y[i : i + 1],
                    device=device,
                )
                s = float(torch.dot(g, vg).item()) if g.numel() == vg.numel() else 0.0
                cur[int(sid[i].item())] = s
                c += 1
                if c >= max_samples:
                    break
            if c >= max_samples:
                break
        stage_scores.append(cur)
    stage1 = stage_scores[0] if len(stage_scores) > 0 else {}
    stage2 = stage_scores[1] if len(stage_scores) > 1 else {}
    ids = set(stage1.keys()) | set(stage2.keys())
    total = {k: stage1.get(k, 0.0) + stage2.get(k, 0.0) for k in ids}
    return total, stage1, stage2

### resolve_quick_run_file

> 作用：根据 quick_run 数据集名称解析文件路径、分隔符和是否含显式评分字段。

In [38]:
def resolve_quick_run_file(repo_root: Path, quick_dataset: str):
    ds = quick_dataset.strip().lower()
    base = Path(repo_root) / "recacc" / "quick_run_data"
    if ds == "pinterest":
        return base / "Pinterest" / "pinterest_data.csv", ",", False
    if ds == "yahoo":
        return base / "Yahoo" / "Yahoo_ratings.csv", ",", True
    if ds in ["ml1m", "movielens", "movie_lens"]:
        return base / "ML1M" / "ratings.dat", "::", True
    raise ValueError(f"Unsupported quick dataset: {quick_dataset}")

### parse_quick_run_data

> 作用：解析 quick_run 数据并输出 PreparedData 与 SPINRec 风格 artifact 字典（含 required_num_buckets）。

In [ ]:
def parse_quick_run_data(
    data_file: str,
    quick_dataset: str = "Pinterest",
    max_rows: int = 20000,
    num_buckets: int = 50000,
    sep: str = ",",
    has_rating: bool = False,
    seed: int = 42,
    min_items_per_user: int = 2,
    min_users_per_item: int = 2,
    return_artifacts: bool = True,
):
    del sep, has_rating  # dataset-specific parsing is handled in parse_quick_dataset
    raw = parse_quick_dataset(data_file, dataset=quick_dataset)
    if max_rows is not None and max_rows > 0 and len(raw) > max_rows:
        raw = raw.head(max_rows).copy()

    prepared, artifacts0, stats = prepare_quick_run_data_spinrec_style(
        quick_data_path=data_file,
        dataset_name=quick_dataset,
        min_user_count=min_items_per_user,
        min_item_count=min_users_per_item,
        user_test_ratio=0.2,
        neg_per_pos=5,
        train_neg_ratio = 1,
        seed=seed,
    )

    num_users = int(stats["num_users"])
    num_items = int(stats["num_items"])

    train_df = artifacts0["train_df_uid_iid"]
    test_df = artifacts0["test_df_uid_iid"]

    train_matrix = np.zeros((num_users, num_items), dtype=np.int8)
    if len(train_df) > 0:
        train_matrix[train_df["uid"].values, train_df["iid"].values] = 1

    test_matrix = np.zeros((num_users, num_items), dtype=np.int8)
    if len(test_df) > 0:
        test_matrix[test_df["uid"].values, test_df["iid"].values] = 1

    static_test_matrix = test_matrix.copy()
    static_test_pos = artifacts0["test_pos_pairs"]
    static_test_neg = artifacts0["test_neg_pairs"]
    pop_array = artifacts0["item_popularity"]

    required_num_buckets = int(max(num_buckets, num_users + 1, num_items + 1))

    artifacts = {
        "num_users": num_users,
        "num_items": num_items,
        "train_matrix": train_matrix,
        "test_matrix": test_matrix,
        "static_test_matrix": static_test_matrix,
        "static_test_pos": static_test_pos,
        "static_test_neg": static_test_neg,
        "pop_array": pop_array,
        "required_num_buckets": required_num_buckets,
    }

    if return_artifacts:
        return prepared, artifacts
    return prepared

### 最终覆盖修复（tuple 输出兼容）

 > 作用：确保 VAE 的 (logit, mu, logvar) 输出在估值链路中被正确处理，避免 tuple.view 报错。

In [40]:
def _extract_logits(out):
    if isinstance(out, (tuple, list)):
        return out[0]
    return out


def sample_grad(model: nn.Module, dense: torch.Tensor, sparse: Dict[str, torch.Tensor], y: torch.Tensor, device: str = DEVICE) -> torch.Tensor:
    model.eval()
    dense = dense.to(device)
    sparse = {k: v.to(device) for k, v in sparse.items()}
    y = y.to(device)
    logits = _extract_logits(model(dense, sparse))
    loss = _loss(logits, y)
    grads = torch.autograd.grad(loss, [p for p in model.parameters() if p.requires_grad], retain_graph=False, allow_unused=True)
    return _flat([x.detach().cpu() for x in grads if x is not None])


def val_grad(model: nn.Module, loader: DataLoader, device: str = DEVICE) -> torch.Tensor:
    model.eval()
    acc = None
    n = 0
    for batch in loader:
        dense, sparse, y, _sid = _unpack_batch(batch)
        dense = dense.to(device)
        sparse = {k: v.to(device) for k, v in sparse.items()}
        y = y.to(device)
        logits = _extract_logits(model(dense, sparse))
        loss = _loss(logits, y)
        grads = torch.autograd.grad(loss, [p for p in model.parameters() if p.requires_grad], retain_graph=False, allow_unused=True)
        g = _flat([x.detach().cpu() for x in grads if x is not None])
        acc = g if acc is None else (acc + g)
        n += 1
    if acc is None:
        return torch.zeros(1)
    return acc / max(n, 1)
